# Workbook Details

**Student Name**: Gerard O'Rourke

**Student Number**: 24514772

*Workbook Structure*


*Comments*



# MN5162 Section 3 Assignment

So far in this module you will have been introduced to the power of Transformer architectures, tools and code on a model hub, named entity recognition, text generation, summarization and question answering and have exercised their uses.

Your assignment task is as follows:

**Pick two topics relating to your company's business that could be addressed or engaged with constructively using the knowledge and skills that you have developed from this module, specifically using two examples of your own e-tivity material and experiences on this module and direct specific references to the course material to make the arguments for it.**

You should indicate the:
- considerations,
- risks,
- expected benefits,
- and the relationship with your proposals to the state of the art in these technologies.

Your arguments should be expressed to convince corporate management that an investigative AI project is worthwhile.


Your submission should contain:

- Statements of two business topics / business processes in your company that could benefit from the domains elaborated in this module
- Elaborations on how the material in this module could be applied to address these issues through an small investigative project
- Discussions of the:
    - costs,
    - benefits,
    - opportunities,
    - risks
    - and trade-offs involved using such a small investigative project

- Examples for your own e-tivity material (some code snippets with corresponding outputs from your e-tivity work on this module) demonstrating utility of the technology to the two topics

- Elaboration on how a system arising from a small investigative project can be:
    - extended into the company's business processes and
    - maintained/serviced/upgraded in the medium to longer term,
    - along with the considerations and implications of this.

Please do not disclose any **confidential** or **potentially sensitive information** in this exercise.

The mindset in this activity is that you want to make a case which convinces your employer to demonstrate your skills.

If you can't or dont' feel comfortable using your company as an example, you can simply make up a company and related usecases/scenarios that are addressed by this assignment.

Refer to the Rubric for Task for grading details

- In order to accommodate any code snippets, please submit your work as a Juypter notebook.
- Your report should be well-structured and well-directed, with corresponding headings so that the distinct sections are obvious and it makes easy reading for a manager
  - for example it should be easy for a manager to read through the content without getting "stuck" in technical detail and code examples should be elaborated in clear language.


# AI investigation project for Acme systems
To investigate how AI could be introduced into our organisation, I propose that we take two processes and investigate how AI could be used to improve these processes.  These are:

1. Bug report analysis and classification
1. Customer Response (text generation) once bugs have been resolved.

## Bug Analysis
Bug analysis is challenging for the following reasons:

- Inconsistent Quality of bug reports

    Reports can vary in clarify and completeness.  Users may provide limited detail, unclear descriptions or use non standard language, particularily in cases where english is not their first language.  This can make accurate intepretation difficult and time consuming.


- Lack of standardisation across channels

    Bug reports may be received in different channels (email, customer call, web).  This leads to unstructured and inconsistent reports, which makes it difficult to extract key information efficiently.

- High / Varying volumes of bug reports

    The number of bug reports can vary, and will typically increase after a release.  This may mean that not every report will get a detailed review, and as a result important issues may be missed.


If AI techniques such as summarisation could be applied to these challenges, it has the potential to improve the efficiency of the bug analysis process.  

Developers would spend less time interepting bug reports allowing for a faster resolution time and overall improved productivity.


## Customer Response Generation (post bug resolution)
Once a bug has been resolved, the customer will need to be notified of this.  Creating a suitable response to a bug report is often a challenge due to:

- Response needs to be in language understood by the customer

    The developer will detail the fix as a series of technical changes (configuration files, software changes).  These will need to be translated into language that the customer will understand, which can be time consuming, when performed manually.

- Consistency of messaging

    Different customer support agaents, will describe the same issue in different ways.  This can lead to differences in the level of details, clarity and tone, which can affect how the customer perceives the response.

- Time pressure

    Once an issue is resolved, the customer response should be generated quickly.  However, this response may be delayed if the developer is dealing with multiple issues.

If AI could be applied to this process, to generate a draft response to the customer, the customer support agent could review and refine the response before it is sent to the customer.  

This would:

1. Reduce the time required to communicate a resolution to a customer
1. Improve consistency across responses
1. Ensure solutions are clearly communicated in a customer friendly language.

# Bug Analysis

In [5]:
from transformers import pipeline
import pandas as pd
import textwrap
from pprint import pprint
import requests

## Example bug reports
These are a mixture of real bug reports from VSCode and a report generated by hand and a long one from ChatGpt.

In [28]:
file_urls = ['https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_1_312390.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_2_312388.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_3_312385.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_4_312384.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_5_312365.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/good_bug_text_1.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/long_bug_text_1.txt',
             'https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/long_bug_text_2.txt']

## Build Summarisation Model

In [2]:
# summary from assignment 1
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load BART for summarization
summary_model_name = "facebook/bart-large-cnn"
summary_tokenizer = AutoTokenizer.from_pretrained(summary_model_name)
summary_model = AutoModelForSeq2SeqLM.from_pretrained(summary_model_name)

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [3]:
def summarise_text(model, tokenizer, text):

    # Generate summary
    inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
    summary_ids = model.generate(
        **inputs,
        max_new_tokens=150,      # Use max_new_tokens, not max_length
        min_length=30,
        length_penalty=2.0,
        num_beams=4,
        early_stopping=True
    )

    summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    #print("Summary:", summary)

    return summary

### Retrieve bug reports

In [16]:
from dataclasses import dataclass
from typing import Any

@dataclass
class BugSummary:
    original_text: str
    url : str
    normalised_letter:str = None
    summary: str = None
    summary_gen_time : float = 0;


In [29]:
bugs = []

url_data = [];

for url in file_urls:
    # don't cache file, while testing
    headers={"Cache-Control":"no-cache"}
    bug_text = requests.get(url, headers=headers).text

    url_data.append({'url':url, 'bug_text':bug_text})

    record = BugSummary( bug_text, url)

    bugs.append(record)


df = pd.DataFrame(url_data)
pd.set_option('display.max_colwidth', 600)
df[['url','bug_text']]



,url,bug_text
0,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_1_312390.txt,"Link - https://github.com/microsoft/vscode/issues/312390\n\nDoes this issue occur when all extensions are disabled?: Yes\n\nVS Code Version: 1.118.0-insiders\nOS Version: Windows 11 25R2\nSteps to Reproduce:\n\nOpen VS Code Agents\nAsk some copilot-type work\nObserve that the ""Thinking"" content doesn't appear anymore\nObserve that you can't see any of the tool calls anymore, no ""Editing file"", nothing\nThis makes the entire app a black box, impossible to know what's happening at all. A couple days ago it had all the information I could need, I could open any tool calls and see the inputs/o..."
1,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_2_312388.txt,"Link https://github.com/microsoft/vscode/issues/312388\nType: Feature Request\n\nLSP-to-MCP\nHow do you feel\n\nVS Code version: Code 1.117.0 (10c8e55, 2026-04-21T16:12:14-07:00)\nOS version: Windows_NT x64 10.0.26200\nModes:\n"
2,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_3_312385.txt,"https://github.com/microsoft/vscode/issues/312385\nType: Bug\n\ni want to use copilot on github and i need to sign in ,its giving me problem when i want to sign in ,kindly help\n\nExtension version: 0.44.2\nVS Code version: Code 1.116.0 (560a9db, 2026-04-15T00:28:13Z)\nOS version: Windows_NT x64 10.0.19045\nModes:\n\nLogs\nError: Error: GitHubLoginFailed\n\tat Are._authShowWarnings (c:\Users\hp\.vscode\extensions\github.copilot-chat-0.44.2\dist\extension.js:1721:18537)\n\tat async Are.getCopilotToken (c:\Users\hp\.vscode\extensions\github.copilot-chat-0.44.2\dist\extension.js:1721:16255)\n..."
3,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_4_312384.txt,"https://github.com/microsoft/vscode/issues/312384\n\nProblem\n\nIn terminalbench evaluation runs, 7 tasks hung indefinitely (3600s timeout) because run_in_terminal never completed. All failed with X_AGENT_STILL_RESPONDING.\n\nAffected tasks\n\nTask\tRun\tWhat happened\ncsv-to-parquet\t24890545491\tAgent ran python3 (bare REPL), tool blocked forever\nbuild-initramfs-qemu\t24890545491\tCommand hung, no shell integration sequences\ncobol-modernization\t24890545491\tLong compilation, shell integration failed\nchess-best-move\t24890545491\tShell integration failure, tool never returned\ncrack-7..."
4,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_5_312365.txt,"https://github.com/microsoft/vscode/issues/312365\n\nCopilot Chat Extension Version: It just says ""Built-in"" now, and I don't see a version. Changelog says 0.41 but that's a month old.\nVersion: 1.118.0-insider\nCommit: 7f677cd\nDate: 2026-04-24T07:39:11Z\nElectron: 39.8.8\nElectronBuildId: 13870025\nChromium: 142.0.7444.265\nNode.js: 22.22.1\nV8: 14.2.231.22-electron.0\nOS: Linux x64 6.8.0-110-generic\n\nSteps to Reproduce:\n\nStart from an existing chat.\nC-p ""Open in Agents"" RET\nNothing happens, in Output window Main logs this line:\n2026-04-24 08:01:35.044 [warning] [launchSiblingApp]..."
5,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/good_bug_text_1.txt,"Hi,\n\nI am trying to use your banking application to do an intraday transfer to Germany. The system is allowing me enter a valid IBAN number but when I click transfer, the transfer does not complete. Instead it reports that the BIC code is incorrect. However that does not make sense as I'm not using a BIC code to do the transfer. I am using the IBAN which the system has previously verified as correct. Can you address this urgently as I need to make the payment by 3pm today.\n\nPlease telephone me if you need any additional information regarding this request.\n\nRegards,\n\nSanta.\n"
6,https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/long_bug_text_1.txt,"https://github.com/microsoft/vscode/issues/293405\n\nIt has been nearly three years since the release of the Dark/Light Modern themes, and the developer tools landscap

In [30]:
import time

for bug in bugs:

    print (f"Rec {bug.url}")

    #feedback.normalised_letter = preprocess_letter_text_stage1(feedback.original_letter)


    # get customer name before changing case, as that affects its performance
    #feedback.customer_name = extract_customer_name(ner_pipeline, feedback.original_letter)

    # generate summary
    startTime = time.time();

    summary_text = summarise_text(summary_model, summary_tokenizer, bug.original_text)

    end = time.time();

    bug.summary_gen_time = end - startTime
    bug.summary = textwrap.fill(summary_text, width=80)

Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_1_312390.txt
Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_2_312388.txt
Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_3_312385.txt
Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_4_312384.txt
Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_5_312365.txt
Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/good_bug_text_1.txt
Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/long_bug_text_1.txt
Rec https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/long_bug_text_2.txt


In [31]:
df = pd.DataFrame(bugs)
#df['filename'] = df['url'].apply(lambda x: x[-5:])
#df_for_display = df.drop(columns='url')
#df_columns = df.columns.tolist()

#df_columns = df_columns[-1:] + df_columns[:-1]
#df_for_display = df[df_columns]
display(df)

,original_text,url,normalised_letter,summary,summary_gen_time
0,"Link - https://github.com/microsoft/vscode/issues/312390\n\nDoes this issue occur when all extensions are disabled?: Yes\n\nVS Code Version: 1.118.0-insiders\nOS Version: Windows 11 25R2\nSteps to Reproduce:\n\nOpen VS Code Agents\nAsk some copilot-type work\nObserve that the ""Thinking"" content doesn't appear anymore\nObserve that you can't see any of the tool calls anymore, no ""Editing file"", nothing\nThis makes the entire app a black box, impossible to know what's happening at all. A couple days ago it had all the information I could need, I could open any tool calls and see the inputs/o...",https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_1_312390.txt,None,"This makes the entire app a black box, impossible to know what's happening at\nall. A couple days ago it had all the information I could need, I could open any\ntool calls and see the inputs/outputs of it. Now this is not usable and I have\nto switch to the CLI again.",19.528936
1,"Link https://github.com/microsoft/vscode/issues/312388\nType: Feature Request\n\nLSP-to-MCP\nHow do you feel\n\nVS Code version: Code 1.117.0 (10c8e55, 2026-04-21T16:12:14-07:00)\nOS version: Windows_NT x64 10.0.26200\nModes:\n",https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_2_312388.txt,None,Link https://github.com/Microsoft/vscode/issues/312388. Type: Feature Request.\nLSP-to-MCPHow do you feel about this?,16.776277
2,"https://github.com/microsoft/vscode/issues/312385\nType: Bug\n\ni want to use copilot on github and i need to sign in ,its giving me problem when i want to sign in ,kindly help\n\nExtension version: 0.44.2\nVS Code version: Code 1.116.0 (560a9db, 2026-04-15T00:28:13Z)\nOS version: Windows_NT x64 10.0.19045\nModes:\n\nLogs\nError: Error: GitHubLoginFailed\n\tat Are._authShowWarnings (c:\Users\hp\.vscode\extensions\github.copilot-chat-0.44.2\dist\extension.js:1721:18537)\n\tat async Are.getCopilotToken (c:\Users\hp\.vscode\extensions\github.copilot-chat-0.44.2\dist\extension.js:1721:16255)\n...",https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_3_312385.txt,None,"Extension version: 0.44.2VS Code version: Code 1.116.0 (560a9db,\n2026-04-15T00:28:13Z) OS version: Windows_NT x64 10.0.19045Modes: macros.logs.",36.358699
3,"https://github.com/microsoft/vscode/issues/312384\n\nProblem\n\nIn terminalbench evaluation runs, 7 tasks hung indefinitely (3600s timeout) because run_in_terminal never completed. All failed with X_AGENT_STILL_RESPONDING.\n\nAffected tasks\n\nTask\tRun\tWhat happened\ncsv-to-parquet\t24890545491\tAgent ran python3 (bare REPL), tool blocked forever\nbuild-initramfs-qemu\t24890545491\tCommand hung, no shell integration sequences\ncobol-modernization\t24890545491\tLong compilation, shell integration failed\nchess-best-move\t24890545491\tShell integration failure, tool never returned\ncrack-7...",https://raw.githubusercontent.com/qaz73/MN5162/main/input_data/bug_4_312384.txt,None,"In terminalbench evaluation runs, 7 tasks hung indefinitely (3600s timeout)\nbecause run_in_terminal never completed. All failed with\nX_AGENT_STILL_RESPONDING. The hang originates in trackIdleOnPrompt()\n(executeStrategy.ts:162-225), which is used as a fallback in the Promise.race in\nboth RichExecuteStrategy and BasicExecute Strategy.",46.257329
4,"https://github.com/microsoft/vscode/issues/312365\n\nCopilot Chat Extension Version: It just says ""Built-in"" now, and I don't see a version. Changelog says 0.41 but that's a month old.\nVersion: 1.118.0-insider\nCommit: 7f677cd\nDate: 2026-04-24T07:39:11Z\nElectron: 39.8.8\nElectronBuildId: 13870025\nChromium: 142.0.7444.265\nNode.js: 22.22.1\nV8: 14.2.231.22-electron.0\nOS: Linux x64 6.8.0-110-generic\n\nSteps to Reproduce:\n\nStart from an existing chat.\nC-p ""Open in Agents"" RET\nNothing happens, in Output window Main logs this line:\n2026-04-24 08:01:35.044 [warning] [launchSiblingApp]...",https://raw.githubusercon